In [65]:
import pandas as pd
import numpy as np
import torch
from torchvision.models import resnet50
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
from tqdm.notebook import tqdm

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

transformTrain = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomGrayscale(),
    transforms.ColorJitter(0.05,0.07,0.01,0.06),
    transforms.RandomInvert(),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])
])
transformTest = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean = [0.485, 0.456, 0.406],std = [0.229, 0.224, 0.225])
])

class LoadingSofiane(Dataset):
    def __init__(self,df,transforms,is_train):
        self.df = df
        self.transforms = transforms
        self.is_train = is_train
    def __len__(self):
        return len(self.df)
    def __getitem__(self,idx):
        img = Image.open(self.df['ImagePath'].iloc[idx])
        if img.mode == 'P':
            img = img.convert('RGBA')
        img = self.transforms(img.convert('RGB'))
        if self.is_train:
            return img, torch.tensor(self.df['Label'].iloc[idx], dtype = torch.long)
        else:
            return img

trd = LoadingSofiane(train_df,transformTrain, True)
tsd = LoadingSofiane(test_df, transformTest, False)

train = DataLoader(trd,32,True)
test = DataLoader(tsd,1,False)

In [67]:
model = resnet50(weights='DEFAULT')
model.fc = torch.nn.Sequential(
    torch.nn.Linear(model.fc.in_features, 2)
)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr = 1e-4)

In [68]:
def train_one_epoch():
    model.train()
    running_loss = 0.0
    pbar = tqdm(train)
    
    for images, masks in pbar:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        pbar.set_postfix(loss=loss.item())
        
    epoch_loss = running_loss / len(train.dataset)
    return epoch_loss

In [69]:
for i in range(0, 5):
    print(f'EPoch is {i}')
    loss = train_one_epoch()
    print(f'loss is {loss}')

EPoch is 0


  0%|          | 0/25 [00:00<?, ?it/s]

loss is 0.6533993274666542
EPoch is 1


  0%|          | 0/25 [00:00<?, ?it/s]

loss is 0.46391801224198453
EPoch is 2


  0%|          | 0/25 [00:00<?, ?it/s]

loss is 0.3089816105920215
EPoch is 3


  0%|          | 0/25 [00:00<?, ?it/s]

loss is 0.19453529574612313
EPoch is 4


  0%|          | 0/25 [00:00<?, ?it/s]

loss is 0.1329990756681107


In [70]:
answer = []
model.eval()
with torch.no_grad():
    bar = tqdm(test)
    for i in bar:
        out = model(i)
        answer.append(torch.argmax(out).cpu().numpy())

  0%|          | 0/194 [00:00<?, ?it/s]

In [71]:
pd.DataFrame({
    'SampleID':test_df['SampleID'].tolist(),
    'Label':np.array(answer).squeeze()
}).to_csv('submission.csv',index=False)